In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root.name != "big_data_assignment" and project_root.parent != project_root:
    project_root = project_root.parent

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print("Using project root:", project_root)
print("Ensured project root on sys.path")

### Environment setup

Before running this notebook, install the project dependencies in a terminal from the project root (`big_data_assignment`):

```bash
cd ~/Desktop/Big-Data/big_data_assignment
pip install -r requirements.txt
```

After installing, restart the Jupyter kernel and re-run the first environment cell.

# IMDB Train Data - Completeness & Quality Checks

This notebook loads the IMDB training data and performs:
- **Datatype check per column**
- **Missing value analysis**
- **Disguised missing data detection**
- **Checks for negative numeric values in `runtimeMinutes` and `numVotes`**


In [105]:
import pandas as pd
from pathlib import Path

# Path to the IMDB project directory
base_path = Path('..') / 'big-data-course-2024-projects' / 'imdb'

# Collect and load all train-*.csv files
train_files = sorted([f for f in base_path.glob('train-*.csv') if f.is_file()])
dfs = []
for f in train_files:
    print(f'Loading {f.name}...')
    dfs.append(pd.read_csv(f))

df_train = pd.concat(dfs, ignore_index=True)
print(f'Total rows in combined train data: {len(df_train)}')
df_train.head()

Loading train-1.csv...
Loading train-2.csv...
Loading train-3.csv...
Loading train-4.csv...
Loading train-5.csv...
Loading train-6.csv...
Loading train-7.csv...
Loading train-8.csv...
Total rows in combined train data: 7959


,Unnamed: 0,tconst,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,numVotes,label
0,4,tt0010600,The Doll,Die Puppe,1919,\N,66,1898.0,True
1,7,tt0011841,Way Down East,Way Down East,1920,\N,145,5376.0,True
2,9,tt0012494,Déstiny,Der müde Tod,1921,\N,97,5842.0,True
3,25,tt0015163,The Navigator,The Navigator,1924,\N,59,9652.0,True
4,38,tt0016220,The Phantom of the Opera,The Phantom of the Opera,1925,\N,93,17887.0,True


In [106]:
# 1. Datatype check per column
print("Column dtypes in df_train:\n")
print(df_train.dtypes)


Column dtypes in df_train:

Unnamed: 0          int64
tconst             object
primaryTitle       object
originalTitle      object
startYear          object
endYear            object
runtimeMinutes     object
numVotes          float64
label                bool
dtype: object


In [107]:
# Initial DF: check if startYear, endYear, runtimeMinutes, numVotes have values represented as strings

cols_check = ["startYear", "endYear", "runtimeMinutes", "numVotes"]

print("String representation check on initial df_train:\n")
print("Column            | dtype    | stored as string? | non-numeric string count | examples (non-numeric)")
print("-" * 95)

for col in cols_check:
    if col not in df_train.columns:
        continue
    s = df_train[col]
    is_object = s.dtype == "object" or s.dtype.name == "object"
    stored_as_string = "Yes" if is_object else "No (numeric dtype)"

    if is_object:
        coerced = pd.to_numeric(s, errors="coerce")
        # Non-numeric = non-null in original but NaN after coercion
        non_numeric = s.notna() & coerced.isna()
        n_non_numeric = non_numeric.sum()
        examples = s[non_numeric].drop_duplicates().head(5).tolist() if n_non_numeric > 0 else []
        examples_str = str(examples) if examples else "—"
    else:
        n_non_numeric = 0
        examples_str = "—"

    print(f"{col:17} | {str(s.dtype):8} | {stored_as_string:18} | {n_non_numeric:24} | {examples_str}")

String representation check on initial df_train:

Column            | dtype    | stored as string? | non-numeric string count | examples (non-numeric)
-----------------------------------------------------------------------------------------------
startYear         | object   | Yes                |                      786 | ['\\N']
endYear           | object   | Yes                |                     7173 | ['\\N']
runtimeMinutes    | object   | Yes                |                       13 | ['\\N']
numVotes          | float64  | No (numeric dtype) |                        0 | —


In [108]:
# 2. Missing values per column (standard NaNs)
missing_counts = df_train.isna().sum()
missing_ratio = df_train.isna().mean()

print("Missing values per column:\n")
print(missing_counts)
print("\nMissing ratio per column:\n")
print(missing_ratio)


Missing values per column:

Unnamed: 0           0
tconst               0
primaryTitle         0
originalTitle     3988
startYear            0
endYear              0
runtimeMinutes       0
numVotes           790
label                0
dtype: int64

Missing ratio per column:

Unnamed: 0        0.000000
tconst            0.000000
primaryTitle      0.000000
originalTitle     0.501068
startYear         0.000000
endYear           0.000000
runtimeMinutes    0.000000
numVotes          0.099259
label             0.000000
dtype: float64


In [109]:
# 2b. Recompute missingness after treating '\\N' as NaN

import numpy as np

# Create a copy where IMDB-style null token '\\N' is treated as missing
# We only apply this to object (string-like) columns

df_train_imputed = df_train.copy()
for col in df_train_imputed.columns:
    if df_train_imputed[col].dtype == "O":
        df_train_imputed[col] = df_train_imputed[col].replace("\\N", np.nan)

missing_counts_imputed = df_train_imputed.isna().sum()
missing_ratio_imputed = df_train_imputed.isna().mean()

print("Missing values per column AFTER imputing \\N as NaN:\n")
print(missing_counts_imputed)
print("\nMissing ratio per column AFTER imputing \\N as NaN:\n")
print(missing_ratio_imputed)

Missing values per column AFTER imputing \N as NaN:

Unnamed: 0           0
tconst               0
primaryTitle         0
originalTitle     3988
startYear          786
endYear           7173
runtimeMinutes      13
numVotes           790
label                0
dtype: int64

Missing ratio per column AFTER imputing \N as NaN:

Unnamed: 0        0.000000
tconst            0.000000
primaryTitle      0.000000
originalTitle     0.501068
startYear         0.098756
endYear           0.901244
runtimeMinutes    0.001633
numVotes          0.099259
label             0.000000
dtype: float64


In [110]:
# After imputing \N as NaN: check if startYear, endYear, runtimeMinutes, numVotes still have values as strings

cols_check = ["startYear", "endYear", "runtimeMinutes", "numVotes"]

print("String representation check on df_train_imputed (after \\N → NaN):\n")
print("Column            | dtype    | stored as string? | non-numeric string count | examples (non-numeric)")
print("-" * 95)

for col in cols_check:
    if col not in df_train_imputed.columns:
        continue
    s = df_train_imputed[col]
    is_object = s.dtype == "object" or s.dtype.name == "object"
    stored_as_string = "Yes" if is_object else "No (numeric dtype)"

    if is_object:
        coerced = pd.to_numeric(s, errors="coerce")
        non_numeric = s.notna() & coerced.isna()
        n_non_numeric = int(non_numeric.sum())
        examples = s[non_numeric].drop_duplicates().head(5).tolist() if n_non_numeric > 0 else []
        examples_str = str(examples) if examples else "—"
    else:
        n_non_numeric = 0
        examples_str = "—"

    print(f"{col:17} | {str(s.dtype):8} | {stored_as_string:18} | {n_non_numeric:24} | {examples_str}")

String representation check on df_train_imputed (after \N → NaN):

Column            | dtype    | stored as string? | non-numeric string count | examples (non-numeric)
-----------------------------------------------------------------------------------------------
startYear         | object   | Yes                |                        0 | —
endYear           | object   | Yes                |                        0 | —
runtimeMinutes    | object   | Yes                |                        0 | —
numVotes          | float64  | No (numeric dtype) |                        0 | —


In [111]:
# Outlier detection for numeric columns (df_train_imputed)
# Range + robust statistics (median, trimmed mean, MAD); MAD-based outlier rule

import numpy as np

numeric_cols = ["startYear", "endYear", "runtimeMinutes", "numVotes"]
# k tunes sensitivity: larger k = fewer outliers (e.g. k=3 is conservative)
k_mad = 3
# alpha = fraction to trim from each tail for trimmed mean (e.g. 0.05 = 5% each side)
alpha_trim = 0.05

def mad(x):
    """Median Absolute Deviation: median of |x - median(x)|."""
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan
    return np.median(np.abs(x - np.median(x)))

def trimmed_mean(x, alpha):
    """Drop top and bottom alpha fraction, then mean of the rest."""
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan
    n = len(x)
    trim_count = max(1, int(np.ceil(n * alpha)))
    if 2 * trim_count >= n:
        return np.nan
    x_sorted = np.sort(x)
    return np.mean(x_sorted[trim_count : n - trim_count])

print("=== OUTLIER DETECTION (range + robust statistics) ===\n")

for col in numeric_cols:
    if col not in df_train_imputed.columns:
        continue
    s = pd.to_numeric(df_train_imputed[col], errors="coerce")
    x = s.dropna().values
    n_valid = len(x)
    if n_valid == 0:
        print(f"--- {col}: no non-null values, skipping ---\n")
        continue

    # Range
    x_min, x_max = np.min(x), np.max(x)
    x_range = x_max - x_min
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    lb_iqr = q1 - 1.5 * iqr
    ub_iqr = q3 + 1.5 * iqr

    # Robust center and dispersion
    med = np.median(x)
    tr_mean = trimmed_mean(x, alpha_trim)
    mad_val = mad(x)
    # Scaled MAD (1.4826) approximates sigma for normal data
    scaled_mad = 1.4826 * mad_val if mad_val > 0 else np.nan

    # MAD-based outlier bounds
    lb = med - k_mad * mad_val
    ub = med + k_mad * mad_val
    if mad_val == 0:
        lb, ub = med, med
    is_outlier = (s < lb) | (s > ub)
    n_out = int(is_outlier.sum())
    pct_out = 100 * n_out / n_valid if n_valid else 0

    print(f"--- {col} ---")
    print(f"  Range: min={x_min}, max={x_max}, range={x_range}")
    print(f"  IQR: {iqr:.2f}  |  [Q1-1.5*IQR, Q3+1.5*IQR] = [{lb_iqr:.2f}, {ub_iqr:.2f}]")
    print(f"  Robust center: median={med:.2f}, {alpha_trim*100:.0f}%-trimmed mean={tr_mean:.2f}")
    print(f"  Robust dispersion: MAD={mad_val:.2f}, scaled MAD={scaled_mad:.2f}")
    print(f"  MAD-based bounds (k={k_mad}): [{lb:.2f}, {ub:.2f}]")
    print(f"  Outliers: {n_out} ({pct_out:.2f}% of non-null)")
    if n_out > 0:
        out_vals = s[is_outlier].drop_duplicates().head(10).tolist()
        print(f"  Example outlier values: {out_vals}")
    print()

=== OUTLIER DETECTION (range + robust statistics) ===

--- startYear ---
  Range: min=1918.0, max=2021.0, range=103.0
  IQR: 26.00  |  [Q1-1.5*IQR, Q3+1.5*IQR] = [1949.00, 2053.00]
  Robust center: median=2006.00, 5%-trimmed mean=2000.07
  Robust dispersion: MAD=10.00, scaled MAD=14.83
  MAD-based bounds (k=3): [1976.00, 2036.00]
  Outliers: 1166 (16.26% of non-null)
  Example outlier values: [1919.0, 1920.0, 1921.0, 1924.0, 1925.0, 1926.0, 1929.0, 1933.0, 1934.0, 1936.0]

--- endYear ---
  Range: min=1921.0, max=2021.0, range=100.0
  IQR: 26.00  |  [Q1-1.5*IQR, Q3+1.5*IQR] = [1950.00, 2054.00]
  Robust center: median=2006.50, 5%-trimmed mean=2000.95
  Robust dispersion: MAD=10.50, scaled MAD=15.57
  MAD-based bounds (k=3): [1975.00, 2038.00]
  Outliers: 121 (15.39% of non-null)
  Example outlier values: [1940.0, 1947.0, 1956.0, 1957.0, 1958.0, 1962.0, 1963.0, 1968.0, 1970.0, 1971.0]

--- runtimeMinutes ---
  Range: min=45.0, max=551.0, range=506.0
  IQR: 25.00  |  [Q1-1.5*IQR, Q3+1.5*

In [112]:
# Rows where at least one of startYear, endYear, runtimeMinutes is \N
cols = ["startYear", "endYear", "runtimeMinutes"]
mask = (df_train["startYear"] == "\\N") | (df_train["endYear"] == "\\N") | (df_train["runtimeMinutes"] == "\\N")
examples = df_train.loc[mask, ["tconst", "primaryTitle", "originalTitle", "startYear", "endYear", "runtimeMinutes"]].head(15)
examples

,tconst,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes
0,tt0010600,The Doll,Die Puppe,1919,\N,66
1,tt0011841,Way Down East,Way Down East,1920,\N,145
2,tt0012494,Déstiny,Der müde Tod,1921,\N,97
3,tt0015163,The Navigator,The Navigator,1924,\N,59
4,tt0016220,The Phantom of the Opera,The Phantom of the Opera,1925,\N,93
5,tt0016630,Báttling Bútlér,Battling Butler,1926,\N,77
6,tt0021015,Juno and the Paycock,NaN,1929,\N,85
7,tt0023973,Thé Éáglé ánd thé Háwk,NaN,1933,\N,73
8,tt0023986,Émplớyéés' Éntráncé,NaN,1933,\N,75
9,tt0024184,The Invisible Man,The Invisible Man,1933,\N,71


In [113]:
# Initial DF: check if startYear, endYear, runtimeMinutes, numVotes have values represented as strings

cols_check = ["startYear", "endYear", "runtimeMinutes", "numVotes"]

print("String representation check on initial df_train:\n")
print("Column            | dtype    | stored as string? | non-numeric string count | examples (non-numeric)")
print("-" * 95)

for col in cols_check:
    if col not in df_train.columns:
        continue
    s = df_train[col]
    is_object = s.dtype == "object" or s.dtype.name == "object"
    stored_as_string = "Yes" if is_object else "No (numeric dtype)"

    if is_object:
        coerced = pd.to_numeric(s, errors="coerce")
        # Non-numeric = non-null in original but NaN after coercion
        non_numeric = s.notna() & coerced.isna()
        n_non_numeric = non_numeric.sum()
        examples = s[non_numeric].drop_duplicates().head(5).tolist() if n_non_numeric > 0 else []
        examples_str = str(examples) if examples else "—"
    else:
        n_non_numeric = 0
        examples_str = "—"

    print(f"{col:17} | {str(s.dtype):8} | {stored_as_string:18} | {n_non_numeric:24} | {examples_str}")

String representation check on initial df_train:

Column            | dtype    | stored as string? | non-numeric string count | examples (non-numeric)
-----------------------------------------------------------------------------------------------
startYear         | object   | Yes                |                      786 | ['\\N']
endYear           | object   | Yes                |                     7173 | ['\\N']
runtimeMinutes    | object   | Yes                |                       13 | ['\\N']
numVotes          | float64  | No (numeric dtype) |                        0 | —


In [114]:
# Heuristic MCAR diagnostics: do missingness indicators depend on other values?

import numpy as np

mcar_cols = [c for c in ["startYear", "endYear", "runtimeMinutes", "numVotes"] if c in df_train_imputed.columns]

# Numeric-only version of these columns
mcar_numeric = df_train_imputed[mcar_cols].apply(pd.to_numeric, errors="coerce")

# Missingness indicators (1 = missing, 0 = observed)
miss_indicators = mcar_numeric.isna().astype(int).add_suffix("_is_missing")

print("Columns used for MCAR-style diagnostics:", mcar_cols)
print("\nMissingness indicator correlations (high values suggest dependence among missing patterns):\n")
print(miss_indicators.corr())

# For each column, compare means of other variables between missing vs observed
print("\nGroup-wise means of other variables by missing/observed status:\n")
for col in mcar_cols:
    ind_col = col + "_is_missing"
    ind = miss_indicators[ind_col]
    others = mcar_numeric.drop(columns=[col])
    if others.empty:
        continue
    means_observed = others[ind == 0].mean()
    means_missing = others[ind == 1].mean()
    print(f"\n=== Missingness in '{col}' (indicator: {ind_col}) ===")
    print("Means when value is OBSERVED:\n", means_observed)
    print("Means when value is MISSING:\n", means_missing)

Columns used for MCAR-style diagnostics: ['startYear', 'endYear', 'runtimeMinutes', 'numVotes']

Missingness indicator correlations (high values suggest dependence among missing patterns):

                           startYear_is_missing  endYear_is_missing  \
startYear_is_missing                   1.000000           -1.000000   
endYear_is_missing                    -1.000000            1.000000   
runtimeMinutes_is_missing             -0.013389            0.013389   
numVotes_is_missing                    0.007018           -0.007018   

                           runtimeMinutes_is_missing  numVotes_is_missing  
startYear_is_missing                       -0.013389             0.007018  
endYear_is_missing                          0.013389            -0.007018  
runtimeMinutes_is_missing                   1.000000            -0.003021  
numVotes_is_missing                        -0.003021             1.000000  

Group-wise means of other variables by missing/observed status:


=== Mis

**Interpretation of MCAR-style diagnostics**

- **Missingness indicator correlations**
  - **startYear vs endYear (−1.0)**: When `startYear` is missing, `endYear` is almost always observed, and vice versa. That reflects the data structure: movies usually have only `startYear`; series/ongoing titles often have `endYear` and sometimes missing `startYear`. So this missingness is **not MCAR** — it is **structural** (MAR: missingness depends on “type” of title).
  - The other indicator pairs have correlations near 0 (e.g. −0.013, 0.007, −0.003), so those missingness patterns are **not strongly related to each other**.

- **Group-wise means (missing vs observed)**
  - **startYear / endYear**: When one is missing, the other is observed and has a different profile (e.g. mean `numVotes` ~29k vs ~34k). Again, this points to **not MCAR** (missingness tied to whether the title is a movie vs series).
  - **runtimeMinutes**: When runtime is missing, mean `numVotes` is much lower (~1.7k vs ~29.6k) and mean `startYear` is later (~2013 vs ~1998). So **runtime missingness is related to observed variables** (e.g. less popular or newer titles) → **not MCAR**, likely MAR.
  - **numVotes**: Means of `startYear`, `endYear`, and `runtimeMinutes` are very similar between missing and observed groups. **numVotes missingness is the most consistent with MCAR** among the four columns.

**Conclusion:** After treating `\N` as NaN, the missingness in **startYear**, **endYear**, and **runtimeMinutes** appears **not MCAR** (structural or MAR). **numVotes** missingness is the closest to MCAR. For downstream analysis, consider MAR-aware methods (e.g. include predictors of missingness) rather than assuming MCAR everywhere.

In [115]:
# Statistics: primaryTitle vs originalTitle

import numpy as np

df = df_train_imputed if "df_train_imputed" in dir() else df_train
n = len(df)
# For missing-rate: treat \N and empty string as missing when using raw df_train
if "df_train_imputed" not in dir():
    work_pt = df["primaryTitle"].replace(["\\N", ""], np.nan)
    work_ot = df["originalTitle"].replace(["\\N", ""], np.nan)
else:
    work_pt = df["primaryTitle"]
    work_ot = df["originalTitle"]

print("Statistics: primaryTitle vs originalTitle\n")
print(f"Total rows: {n}\n")

# 1. % missing per column
for col, ser in [("primaryTitle", work_pt), ("originalTitle", work_ot)]:
    miss = ser.isna().sum()
    pct_miss = 100 * miss / n if n else 0
    print(f"{col}: {pct_miss:.2f}% missing ({int(miss)} rows)")

# 2. % duplicates (value appears more than once in column)
#    Computed *among non-missing* only, so NaN does not inflate the count.
print()
for col in ["primaryTitle", "originalTitle"]:
    if col not in df.columns:
        continue
    valid = df[col].notna()
    if valid.sum() == 0:
        print(f"{col}: no non-missing values, skip duplicate %")
        continue
    dup = df.loc[valid, col].duplicated(keep=False)
    dup_count = dup.sum()
    pct_dup = 100 * dup_count / n if n else 0
    pct_of_non_missing = 100 * dup_count / valid.sum() if valid.sum() else 0
    print(f"{col}: {pct_dup:.2f}% duplicate rows among non-missing ({int(dup_count)} rows, {pct_of_non_missing:.2f}% of non-missing)")

# 3. % exact same (primaryTitle == originalTitle in same row)
same = (df["primaryTitle"] == df["originalTitle"]) & df["primaryTitle"].notna()
same_count = same.sum()
pct_same = 100 * same_count / n if n else 0
print(f"\nSame row (primaryTitle == originalTitle): {pct_same:.2f}% ({int(same_count)} rows)")

# 4. Overlap: distinct values appearing in both columns (any row)
def _valid(s):
    u = s.dropna().astype(str).str.strip()
    return set(u[u != ""].unique())
set_primary = _valid(df["primaryTitle"])
set_original = _valid(df["originalTitle"])
overlap = set_primary & set_original
n_overlap = len(overlap)
n_primary = len(set_primary)
n_original = len(set_original)
pct_of_primary = 100 * n_overlap / n_primary if n_primary else 0
pct_of_original = 100 * n_overlap / n_original if n_original else 0
print(f"\nDistinct values in both columns (overlap): {n_overlap}")
print(f"  As % of distinct primaryTitle: {pct_of_primary:.2f}% ({n_primary} distinct)")
print(f"  As % of distinct originalTitle: {pct_of_original:.2f}% ({n_original} distinct)")

Statistics: primaryTitle vs originalTitle

Total rows: 7959

primaryTitle: 0.00% missing (0 rows)
originalTitle: 50.11% missing (3988 rows)

primaryTitle: 2.74% duplicate rows among non-missing (218 rows, 2.74% of non-missing)
originalTitle: 0.67% duplicate rows among non-missing (53 rows, 1.33% of non-missing)

Same row (primaryTitle == originalTitle): 31.52% (2509 rows)

Distinct values in both columns (overlap): 2503
  As % of distinct primaryTitle: 31.90% (7846 distinct)
  As % of distinct originalTitle: 63.46% (3944 distinct)


In [116]:
# Duplicate rows: show rows where primaryTitle or originalTitle appears more than once

df = df_train_imputed if "df_train_imputed" in dir() else df_train

# primaryTitle: rows whose primaryTitle appears more than once
dup_primary = df["primaryTitle"].duplicated(keep=False)
print("Duplicate rows for primaryTitle (value appears in more than one row):")
display(df.loc[dup_primary, ["tconst", "primaryTitle", "originalTitle"]].sort_values("primaryTitle"))

# originalTitle: among non-missing only
valid_ot = df["originalTitle"].notna()
dup_original = df.loc[valid_ot, "originalTitle"].duplicated(keep=False)
idx_dup_original = df.loc[valid_ot].index[dup_original]
print("\nDuplicate rows for originalTitle (among non-missing, value appears in more than one row):")
display(df.loc[idx_dup_original, ["tconst", "primaryTitle", "originalTitle"]].sort_values("originalTitle"))

Duplicate rows for primaryTitle (value appears in more than one row):


,tconst,primaryTitle,originalTitle
3511,tt10199664,Adam,Adam
2520,tt1185836,Adam,NaN
5145,tt0086896,Angel,NaN
4452,tt0783767,Angel,NaN
6200,tt0099073,Avalon,NaN
...,...,...,...
2816,tt5615904,Unforgettable,NaN
775,tt3581652,West Side Story,West Side Story
6019,tt0055614,West Side Story,West Side Story
2075,tt0065234,Z,Z



Duplicate rows for originalTitle (among non-missing, value appears in more than one row):


,tconst,primaryTitle,originalTitle
6745,tt2991296,Beneath,Beneath
3690,tt2325518,Bénéáth,Beneath
3254,tt0115783,Bulletproof,Bulletproof
238,tt0094813,Bulletproof,Bulletproof
1579,tt1262877,Coco,Coco
5679,tt2380307,Coco,Coco
51,tt0043918,Thé Littlé Wớrld ớf Dớn Cámillớ,Don Camillo
4140,tt0085454,The World of Don Camillo,Don Camillo
977,tt0027652,Fury,Fury
2695,tt2713180,Fury,Fury


In [117]:
# Sample rows where both startYear and endYear are present (non-missing)

import pandas as pd

df_full = df_train_imputed if "df_train_imputed" in dir() else df_train

# Treat '\\N' as missing if still present
start = pd.to_numeric(df_full["startYear"].replace("\\N", pd.NA), errors="coerce")
end = pd.to_numeric(df_full["endYear"].replace("\\N", pd.NA), errors="coerce")

mask = start.notna() & end.notna()
subset_years = df_full.loc[mask, [
    "tconst", "primaryTitle", "originalTitle", "startYear", "endYear", "runtimeMinutes", "numVotes", "label"
]].copy()

# Show a small sample, sorted by start/end year
subset_years = subset_years.sort_values(["startYear", "endYear"]).head(20)
subset_years

,tconst,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,numVotes,label


In [118]:
# Validity: intra-relation consistency
# Type checks, range constraints, domain validation rules

import re
import numpy as np

# Use post-\N imputation data if available
df = df_train_imputed if 'df_train_imputed' in dir() else df_train

print("=== VALIDITY CHECKS (intra-relation consistency) ===\n")

# ---- 1. Type conformance ----
# Columns that should be numeric but may be stored as object
numeric_cols = ["startYear", "endYear", "runtimeMinutes", "numVotes"]
for col in numeric_cols:
    if col not in df.columns:
        continue
    s = df[col]
    if s.dtype == "object":
        coerced = pd.to_numeric(s, errors="coerce")
        non_numeric = coerced.isna() & s.notna()
        n_bad = non_numeric.sum()
        if n_bad > 0:
            print(f"[TYPE] '{col}': {n_bad} non-numeric values (stored as object)")
            print(f"       Example non-numeric: {s[non_numeric].dropna().head(3).tolist()}")
        else:
            print(f"[TYPE] '{col}': stored as object but all values numeric (after coercion)")
    else:
        print(f"[TYPE] '{col}': dtype {s.dtype} (OK)")

# label: expect bool or binary
if "label" in df.columns:
    unique_labels = df["label"].dropna().unique()
    if not (set(unique_labels) <= {True, False, 1, 0}):
        print(f"[TYPE] 'label': invalid categories {unique_labels.tolist()}")
    else:
        print(f"[TYPE] 'label': valid categories {sorted(set(unique_labels))}")

# ---- 2. Range constraints ----
from datetime import datetime
current_year = datetime.now().year

# startYear: valid film era ~1888 to current year
if "startYear" in df.columns:
    sy = pd.to_numeric(df["startYear"], errors="coerce")
    out_range = (sy.notna()) & ((sy < 1888) | (sy > current_year))
    print(f"\n[RANGE] startYear: {out_range.sum()} values outside [1888, {current_year}]")
    if out_range.any():
        print(f"        Examples: {sy[out_range].dropna().head(5).tolist()}")

# endYear: same range, and should be >= startYear when both present
if "endYear" in df.columns and "startYear" in df.columns:
    ey = pd.to_numeric(df["endYear"], errors="coerce")
    out_range_ey = (ey.notna()) & ((ey < 1888) | (ey > current_year))
    print(f"[RANGE] endYear: {out_range_ey.sum()} values outside [1888, {current_year}]")
    both = sy.notna() & ey.notna()
    invalid_order = both & (ey < sy)
    print(f"[RANGE] endYear < startYear (when both present): {invalid_order.sum()} rows")

# runtimeMinutes: positive, reasonable upper bound (e.g. <= 500)
if "runtimeMinutes" in df.columns:
    rt = pd.to_numeric(df["runtimeMinutes"], errors="coerce")
    out_range_rt = (rt.notna()) & ((rt <= 0) | (rt > 500))
    print(f"[RANGE] runtimeMinutes: {out_range_rt.sum()} values outside (0, 500]")

# numVotes: non-negative
if "numVotes" in df.columns:
    nv = pd.to_numeric(df["numVotes"], errors="coerce")
    print(f"[RANGE] numVotes: {(nv < 0).sum()} values < 0")

# ---- 3. Domain validation ----
# tconst: IMDB format tt + digits (e.g. tt0010600)
if "tconst" in df.columns:
    pattern = re.compile(r"^tt\d+$")
    valid_tconst = df["tconst"].astype(str).str.match(pattern, na=False)
    invalid = (~valid_tconst) & df["tconst"].notna()
    print(f"\n[DOMAIN] tconst: {invalid.sum()} values not matching 'tt' + digits")
    if invalid.any():
        print(f"         Examples: {df.loc[invalid, 'tconst'].head(5).tolist()}")

# primaryTitle / originalTitle: non-empty when present
for col in ["primaryTitle", "originalTitle"]:
    if col not in df.columns:
        continue
    empty = (df[col].astype(str).str.strip() == "") & df[col].notna()
    print(f"[DOMAIN] '{col}': {empty.sum()} empty-or-whitespace values (non-null)")

print("\n--- Validity summary done ---")

=== VALIDITY CHECKS (intra-relation consistency) ===

[TYPE] 'startYear': stored as object but all values numeric (after coercion)
[TYPE] 'endYear': stored as object but all values numeric (after coercion)
[TYPE] 'runtimeMinutes': stored as object but all values numeric (after coercion)
[TYPE] 'numVotes': dtype float64 (OK)
[TYPE] 'label': valid categories [False, True]

[RANGE] startYear: 0 values outside [1888, 2026]
[RANGE] endYear: 0 values outside [1888, 2026]
[RANGE] endYear < startYear (when both present): 0 rows
[RANGE] runtimeMinutes: 1 values outside (0, 500]
[RANGE] numVotes: 0 values < 0

[DOMAIN] tconst: 0 values not matching 'tt' + digits
[DOMAIN] 'primaryTitle': 0 empty-or-whitespace values (non-null)
[DOMAIN] 'originalTitle': 0 empty-or-whitespace values (non-null)

--- Validity summary done ---


In [119]:
# 3. Disguised missing data
# Common disguised tokens in this dataset: '\\N' (IMDB null) and '' (empty string), plus some generic ones.
DISGUISED_TOKENS = ["\\N", "", "NA", "N/A", "null", "None"]

print("Disguised missing values (counts of special tokens) per column:\n")
for col in df_train.columns:
    if df_train[col].dtype == "O":  # only check string-like columns
        vc = df_train[col].value_counts(dropna=False)
        for token in DISGUISED_TOKENS:
            if token in vc.index:
                print(f"Column '{col}' -> token '{token}': {vc[token]} occurrences")

Disguised missing values (counts of special tokens) per column:

Column 'startYear' -> token '\N': 786 occurrences
Column 'endYear' -> token '\N': 7173 occurrences
Column 'runtimeMinutes' -> token '\N': 13 occurrences


In [120]:
# Completeness and missingness analysis (including disguised missing values)

# 1. Standard missing values per column
missing_counts = df_train.isna().sum()
missing_ratio = df_train.isna().mean()

print("Missing values per column:")
print(missing_counts)
print("\nMissing ratio per column:")
print(missing_ratio)

# 2. Disguised missing values: special tokens and empty strings
# Common disguised tokens in this dataset are '\\N' (IMDB null) and '' (empty string).
DISGUISED_TOKENS = ["\\N", "", "NA", "N/A", "null", "None"]

print("\nDisguised missing values (counts of special tokens) per column:")
for col in df_train.columns:
    if df_train[col].dtype == "O":  # only check string-like columns
        vc = df_train[col].value_counts(dropna=False)
        found_any = False
        for token in DISGUISED_TOKENS:
            if token in vc.index:
                print(f"Column '{col}' -> token '{token}': {vc[token]}")
                found_any = True
        if not found_any:
            continue

# 3. Datatype check and suspicious numeric values that may indicate missingness
print("\nColumn dtypes in df_train:\n", df_train.dtypes)

# Coerce numeric-like columns to numeric for safe comparisons
runtime_numeric = None
startyear_numeric = None
if "runtimeMinutes" in df_train.columns:
    runtime_numeric = pd.to_numeric(df_train["runtimeMinutes"], errors="coerce")
if "startYear" in df_train.columns:
    startyear_numeric = pd.to_numeric(df_train["startYear"], errors="coerce")

# Here we inspect zero or negative runtimes and years < 1888 as potential issues.
if runtime_numeric is not None:
    suspect_runtime = df_train[runtime_numeric <= 0]
    print(f"\nRows with non-positive runtimeMinutes: {len(suspect_runtime)}")

if startyear_numeric is not None:
    suspect_year = df_train[startyear_numeric < 1888]
    print(f"Rows with startYear < 1888 (possibly bad or placeholder years): {len(suspect_year)}")

Missing values per column:
Unnamed: 0           0
tconst               0
primaryTitle         0
originalTitle     3988
startYear            0
endYear              0
runtimeMinutes       0
numVotes           790
label                0
dtype: int64

Missing ratio per column:
Unnamed: 0        0.000000
tconst            0.000000
primaryTitle      0.000000
originalTitle     0.501068
startYear         0.000000
endYear           0.000000
runtimeMinutes    0.000000
numVotes          0.099259
label             0.000000
dtype: float64

Disguised missing values (counts of special tokens) per column:
Column 'startYear' -> token '\N': 786
Column 'endYear' -> token '\N': 7173
Column 'runtimeMinutes' -> token '\N': 13

Column dtypes in df_train:
 Unnamed: 0          int64
tconst             object
primaryTitle       object
originalTitle      object
startYear          object
endYear            object
runtimeMinutes     object
numVotes          float64
label                bool
dtype: object

Rows wit

In [121]:
# Utility code to detect disguised missing tokens and suspicious numeric values

import numpy as np

DISGUISED_TOKENS = ["\\N", "", "NA", "N/A", "null", "None"]

# Rules for suspicious numeric values per column
SUSPICIOUS_RULES = {
    # Use pd.to_numeric to safely coerce before comparison
    "runtimeMinutes": lambda s: pd.to_numeric(s, errors="coerce") <= 0,
    # Treat pre-film years as suspicious; coerce non-numeric to NaN first
    "startYear": lambda s: pd.to_numeric(s, errors="coerce") < 1888,
    # You can add more rules here, e.g. very low numVotes
    "numVotes": lambda s: pd.to_numeric(s, errors="coerce") <= 0,
}

# 1. Disguised missing tokens per column
print("Disguised missing tokens per column:\n")
for col in df_train.columns:
    if df_train[col].dtype == "O":  # string-like
        vc = df_train[col].value_counts(dropna=False)
        for token in DISGUISED_TOKENS:
            if token in vc.index:
                print(f"Column '{col}' -> token '{token}': {vc[token]} occurrences")

# 2. Suspicious numeric values based on rules
print("\nSuspicious numeric values by rule:\n")
for col, rule in SUSPICIOUS_RULES.items():
    if col not in df_train.columns:
        continue
    series = df_train[col]
    try:
        mask = rule(series)
    except TypeError:
        # In case dtype conversion failed badly, skip
        continue
    count = int(mask.fillna(False).sum())
    if count > 0:
        print(f"Column '{col}' -> {count} suspicious values")

Disguised missing tokens per column:

Column 'startYear' -> token '\N': 786 occurrences
Column 'endYear' -> token '\N': 7173 occurrences
Column 'runtimeMinutes' -> token '\N': 13 occurrences

Suspicious numeric values by rule:



In [122]:
# Full rows for specific tconsts from the duplicate-rows view
df_full = df_train_imputed if "df_train_imputed" in dir() else df_train

tconsts = ["tt1087856", "tt6816070", "tt0986233", "tt1176252"]

subset = df_full[df_full["tconst"].isin(tconsts)].sort_values("tconst")
subset

,Unnamed: 0,tconst,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,numVotes,label
3501,5555,tt0986233,Hunger,Hunger,2008,NaN,96,69004.0,True
5490,5783,tt1087856,Hello,Hello,NaN,2008,129,2069.0,False
7540,6020,tt1176252,Hunger,Hunger,2009,NaN,101,5101.0,False
4870,9380,tt6816070,Hello!,Hello,2017,NaN,131,2116.0,True


In [123]:
# Full rows for selected tconsts from the duplicate-rows view
df_full = df_train_imputed if "df_train_imputed" in dir() else df_train

tconsts = [
    "tt1087856",  # Hello
    "tt6816070",  # Hello!
    "tt0986233",  # Hunger
    "tt1176252",  # Hunger
    "tt2991296",  # Beneath
    "tt2325518",  # Bénéáth
    "tt0115783",  # Bulletproof
    "tt0094813",  # Bulletproof
    "tt1262877",  # Coco
    "tt2380307",  # Coco
    "tt0043918",  # Thé Littlé Wớrld ớf Dớn Cámillớ
    "tt0085454",  # The World of Don Camillo
    "tt0027652",  # Fury
    "tt2713180",  # Fury
    "tt0339079",  # Gone
    "tt1838544",  # Gớné
    "tt0133826",  # Guru
    "tt0499375",  # Guru
]

subset = df_full[df_full["tconst"].isin(tconsts)].sort_values("tconst")
subset

,Unnamed: 0,tconst,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,numVotes,label
977,165,tt0027652,Fury,Fury,NaN,1936,92,12115.0,True
51,545,tt0043918,Thé Littlé Wớrld ớf Dớn Cámillớ,Don Camillo,1952,NaN,107,4348.0,True
4140,1996,tt0085454,The World of Don Camillo,Don Camillo,1984,NaN,126,1917.0,False
238,2394,tt0094813,Bulletproof,Bulletproof,1987,NaN,93,1344.0,False
3254,3120,tt0115783,Bulletproof,Bulletproof,1996,NaN,84,37755.0,False
5279,3436,tt0133826,Guru,Guru,1997,NaN,136,3065.0,True
2349,4361,tt0339079,Gone,Gone,NaN,2002,80,1362.0,True
498,5209,tt0499375,Guru,Guru,2007,NaN,162,23099.0,True
3501,5555,tt0986233,Hunger,Hunger,2008,NaN,96,69004.0,True
5490,5783,tt1087856,Hello,Hello,NaN,2008,129,2069.0,False


In [125]:
# Sample rows where both startYear and endYear are present (non-missing)

import pandas as pd

# Use the imputed frame if available so '\\N' is already NaN
base_df = df_train_imputed if "df_train_imputed" in dir() else df_train

start = pd.to_numeric(base_df["startYear"], errors="coerce")
end = pd.to_numeric(base_df["endYear"], errors="coerce")

mask = start.notna() & end.notna()
subset_years = base_df.loc[mask, [
    "tconst", "primaryTitle", "originalTitle", "startYear", "endYear", "runtimeMinutes", "numVotes", "label"
]].copy()

# Show a small sample, sorted by start/end year
subset_years.sort_values(["startYear", "endYear"]).head(20)

,tconst,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,numVotes,label


In [ ]:
# Distributional stats for startYear and endYear

import pandas as pd

base_df = df_train_imputed if "df_train_imputed" in dir() else df_train

# Coerce years to numeric, treating '\\N' as missing if still present
sy = pd.to_numeric(base_df["startYear"].replace("\\N", pd.NA), errors="coerce")
ey = pd.to_numeric(base_df["endYear"].replace("\\N", pd.NA), errors="coerce")

print("=== startYear (all non-missing) ===\n")
print(sy.describe())

print("\n=== endYear (all non-missing) ===\n")
print(ey.describe())

# Where both are present (if any)
mask_both = sy.notna() & ey.notna()
print(f"\nRows with both startYear and endYear non-missing: {int(mask_both.sum())}")
if mask_both.any():
    print("\n=== startYear where both present ===\n")
    print(sy[mask_both].describe())
    print("\n=== endYear where both present ===\n")
    print(ey[mask_both].describe())